<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Bioinformatics_DCA/blob/master/FASE_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FASE 3: Traducción + ORFs + BLASTx / BLASTp
# ============================================================
# Objetivo: Encontrar el marco de lectura abierto (ORF) más
# probable y determinar qué proteína codifica (especialmente
# factores de virulencia / toxinas / proteínas de infección).

!pip install -q biopython

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.Blast import NCBIWWW, NCBIXML
from google.colab import files
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# --- 1. Cargar secuencia ---
uploaded = files.upload()
record = SeqIO.read(list(uploaded.keys())[0], "fasta")
nt_seq = record.seq

# --- 2. Traducir en los 6 marcos de lectura y buscar ORFs ---
def find_orfs(seq, min_len=100):
    """Devuelve lista de (frame, start, end, protein_seq)"""
    orfs = []
    for strand, nuc in [(+1, seq), (-1, seq.reverse_complement())]:
        for frame in range(3):
            trans = str(nuc[frame:].translate(to_stop=False))
            # Buscar ORFs que empiecen con M y terminen en stop
            aa_start = 0
            while True:
                start = trans.find("M", aa_start)
                if start == -1:
                    break
                end = trans.find("*", start)
                if end == -1:
                    break
                protein = trans[start:end]
                if len(protein) >= min_len:
                    # Coordenadas en nt
                    nt_start = frame + start*3
                    nt_end   = frame + end*3 + 3
                    if strand == -1:
                        nt_start, nt_end = len(seq)-nt_end, len(seq)-nt_start
                    orfs.append({
                        "strand": strand,
                        "frame": frame+1,
                        "start": nt_start+1,
                        "end": nt_end,
                        "length_aa": len(protein),
                        "seq": protein
                    })
                aa_start = end + 1
    return sorted(orfs, key=lambda x: -x["length_aa"])

orfs = find_orfs(nt_seq, min_len=50)
print(f"Se encontraron {len(orfs)} ORFs ≥ 50 aa\n")
for i, orf in enumerate(orfs[:5], 1):
    print(f"ORF {i}: strand={orf['strand']}, frame={orf['frame']}, "
          f"{orf['start']}-{orf['end']} ({orf['length_aa']} aa)")
    print(f"   Primeros 60 aa: {orf['seq'][:60]}...")
    print()

# --- 3. Elegir el ORF más largo (o el que tenga sentido biológico) ---
best_orf = orfs[0]
protein_seq = best_orf["seq"]
print(f"\n>>> ORF seleccionado: {best_orf['length_aa']} aa")
print(protein_seq)

# --- 4. BLASTp del ORF seleccionado ---
print("\nEnviando BLASTp a NCBI (nr)...")
result_handle = NCBIWWW.qblast(
    program="blastp",
    database="nr",
    sequence=protein_seq,
    hitlist_size=10,
    expect=1e-5
)

with open("blastp_result.xml", "w") as f:
    f.write(result_handle.read())
result_handle.close()

with open("blastp_result.xml") as f:
    blast_record = NCBIXML.read(f)

print(f"\n{'='*60}")
print("Top hits de la proteína:")
print(f"{'='*60}")
for i, alignment in enumerate(blast_record.alignments[:5], 1):
    hsp = alignment.hsps[0]
    identity = (hsp.identities / hsp.align_length) * 100
    print(f"Hit {i}: {alignment.title[:120]}")
    print(f"   Identidad: {identity:.1f}% | E-value: {hsp.expect:.2e}")
    print("-"*50)